# AirBnB Product Project

* By: Nhi Bui

# Statistical Testing for Factor Impact on Review Ratings

## Question 1
What host listing factors controlled by hosts most significantly impact review ratings?

## My proposed approach
Validate EDA findings with statistical tests to determine 
if the observed differences in review ratings across host-controlled 
factors are statistically significant.

In [1]:
#Reload data
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/Airbnb_Open_Data_Cleaned.csv')

# ANOVA Test - Room Type

In [2]:
#Split data into groups by room type
df['room type'].unique()

<StringArray>
['Private room', 'Entire home/apt', 'Shared room', 'Hotel room']
Length: 4, dtype: str

In [3]:
# Create a variable for each group's review ratings
private = df[df['room type'] == 'Private room']['review rate number']
entire = df[df['room type'] == 'Entire home/apt']['review rate number']
shared = df[df['room type'] == 'Shared room']['review rate number']
hotel = df[df['room type'] == 'Hotel room']['review rate number']

In [4]:
#Run ANOVA test to compare the means of the review ratings across the different room types
f_stat, p_value = stats.f_oneway(private, entire, shared, hotel)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

F-statistic: 3.2804
P-value: 0.0200


Because p-value = 0.02 < 0.05
=> There is a statistically significant difference in review ratings between room types
=> Reject the NULL hypothesis
=> This confirms our earlier statement of review rate number affected by different room types

**Post-hoc test**

To find out which specific room types differ from each other, you need a post-hoc test using Tukey's HSD 

In [5]:
#Check for Turkey HSD post-hoc test to see which groups differ
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(df['review rate number'], df['room type'], alpha=0.05)
print(tukey)

       Multiple Comparison of Means - Tukey HSD, FWER=0.05        
     group1        group2    meandiff p-adj   lower  upper  reject
------------------------------------------------------------------
Entire home/apt   Hotel room    0.265 0.1932 -0.0778 0.6079  False
Entire home/apt Private room    0.025 0.0724 -0.0015 0.0514  False
Entire home/apt  Shared room   0.0321  0.801 -0.0588 0.1229  False
     Hotel room Private room  -0.2401 0.2739  -0.583 0.1028  False
     Hotel room  Shared room   -0.233  0.328 -0.5867 0.1208  False
   Private room  Shared room   0.0071 0.9971 -0.0841 0.0983  False
------------------------------------------------------------------


**Conclusion**
* ANOVA confirms a statistically significant difference exists across room types (p = 0.02)
* However, Tukey's post-hoc test shows no single pair is significantly different after correction
* The strongest contrast is between Entire home/apt and Private room (p = 0.0724)
=> This suggests the effect is distributed across groups rather than driven by one standout room type

**Recommendation**

Because the differences in ratings between room types are real but small to justify treating one room type differently from others, Airbnb should focus on improving quality within each room type (amenity checklists, service quality guidelines, professional photo programs, and responsive communication tools) instead of trying to push hosts toward a specific room type 

# ANOVA Test - Price

In [7]:
#Check bins of price
df['price'].dtype

dtype('float64')

Because type of price is float, we need to recreate the bins

In [8]:
#Recreate bins for price
df['price_group'] = pd.cut(df['price'], 
                           bins=[0, 50, 500, 1000, 1500], 
                           labels=['Low', 'Medium', 'High', 'Very High'])

In [9]:
# Create a variable for each group's review ratings
low = df[df['price_group'] == 'Low']['review rate number']
medium = df[df['price_group'] == 'Medium']['review rate number']
high = df[df['price_group'] == 'High']['review rate number']
very_high = df[df['price_group'] == 'Very High']['review rate number']

In [10]:
#Run ANOVA test to compare the means of the review ratings across the different price groups
f_stat, p_value = stats.f_oneway(low, medium, high, very_high)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

F-statistic: 5.5889
P-value: 0.0008


Because p-value = 0.0008 < 0.05
=> There is a statistically significant difference in review ratings between prices
=> Reject the NULL hypothesis
=> This confirms our earlier statement of review rate number affected by different prices

**Post-hoc test**

To find out which specific price levels differ from each other, you need a post-hoc test using Tukey's HSD 

In [11]:
#Check for Turkey HSD post-hoc test to see which groups differ
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(df['review rate number'], df['price_group'], alpha=0.05)
print(tukey)

  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1   group2  meandiff p-adj   lower   upper  reject
-------------------------------------------------------
  High       Low  -0.1429 0.7972 -0.5446  0.2587  False
  High    Medium   0.0153 0.5159 -0.0134   0.044  False
  High Very High  -0.0427 0.0153 -0.0796 -0.0059   True
   Low    Medium   0.1583 0.7422 -0.2434    0.56  False
   Low Very High   0.1002 0.9191 -0.3022  0.5026  False
Medium Very High  -0.0581 0.0004 -0.0955 -0.0206   True
-------------------------------------------------------


**Conclusion**
* Two pairs show reject = True:
    * High vs Very High (p = 0.0153) => significant difference
    * Medium vs Very High (p = 0.0004) => highly significant difference
=> Both involve Very High price being significantly lower in ratings. => This  confirms our earlier EDA hypothesis about the expectation gap
=> Very high priced listings genuinely underperform compared to medium and high priced ones

* Meanwhile, Low price level vs everything shows reject = False
=> The low price effect we saw in EDA wasn't strong enough to be statistically significant


**Recommendation**

1. Pricing guidance for premium hosts
We should implement a pricing quality check that alerts hosts when their price significantly exceeds the neighbourhood average without matching amenity or quality benchmarks

2. Guest expectation management 
For very high priced listings, Airbnb could require more detailed listing descriptions, verified amenity lists, and professional photos to ensure the listing delivers on its price promise

3. Pricing recommendation tool 
Provide hosts with data-driven pricing recommendations that optimize for both revenue and guest satisfaction
=> Guide them toward the medium-to-high range where ratings peak

Given the smaller effect sizes observed in EDA (0.08 points), statistical testing may not needed for availability and minimum nights as they are unlikely to reach significance

However, for reducing risks, we will still conduct tests for these 2 factors

# ANOVA Test - Aavial